In [1]:
#@title 02 - ⚙️ Configuration
from sys import version_info
import os
from pathlib import Path
python_version = f"{version_info.major}.{version_info.minor}"
PROJECT_ROOT = Path.cwd()
TARGET_UNIPROT = "P0ABQ4"  # E. coli K-12 DHFR (folA)
OFFLINE_OUTDIR = PROJECT_ROOT / "data" / TARGET_UNIPROT
OFFLINE_OUTDIR.mkdir(parents=True, exist_ok=True)
CFG = dict(
    UNIPROT_ID=TARGET_UNIPROT,
    LIGAND_SOURCE=dict(kind="pubchem_cid", id="126941"),
    JOB_NAME=f"DHFR_ecoli_{TARGET_UNIPROT}",
    OUTDIR=str(OFFLINE_OUTDIR),
    AFDB_VERSION="v6",
    MODEL_TYPE="alphafold2_ptm",
    NUM_SEEDS=5,
    NUM_RECYCLES=3,
    NUM_MODELS=1,
    NUM_RELAX=1,
    POCKET_CONF=70.0,
    BOX_PADDING=9.0,
    BOOTSTRAP_N=500,
    NUM_SITES=200,
    SHARD_SIZE=120,
    LIGAND_MAX_HEAVY_ATOMS=120,
    LIGAND_MAX_ROTATABLE_BONDS=32,
    LIGAND_MAX_MW=1500.0,
    ALLOW_MACRO_LIGAND=True,
    SWARM_EFFICIENCY_PROFILE="balanced",  # fast | balanced | quality
    SWARM_CLEAN_RECURSIVE_OUTPUTS=False,
    SWARM_RECURSIVE_ARGS={
        # Optional overrides for scripts/swarm/20_run_recursive_adaptive_flow.py
        # Example keys: "panel_total", "proposal_total", "binding_max_variants",

        # "dedupe_scope", "dedupe_lookback_rounds", "binding_score_all",
        # "objective_improvement_eps", "min_budget_fraction_before_voi_stop",
        # "chemistry_coverage_enable", "chemistry_challenger_frac"
    },
)
CFG.update({
    "WT_FASTA": os.path.join(CFG["OUTDIR"], "enzyme_wt.fasta"),
    "REFERENCE_STRUCTURE": os.path.join(CFG["OUTDIR"], f"AF-{CFG['UNIPROT_ID']}-F1-model_{CFG['AFDB_VERSION']}.pdb"),
    "LIGAND_SDF": os.path.join(CFG["OUTDIR"], "ligand.sdf"),
    "LIGAND_PDB": os.path.join(CFG["OUTDIR"], "ligand.pdb"),
    "RECEPTOR_PDBQT": os.path.join(CFG["OUTDIR"], "receptor.pdbqt"),
    "LIGAND_PDBQT": os.path.join(CFG["OUTDIR"], "ligand.pdbqt"),
    "DOCKED_SDF": os.path.join(CFG["OUTDIR"], "docked_top.sdf"),
    "POCKET_FILE": os.path.join(CFG["OUTDIR"], "pocket_residues.txt"),
    "VARIANTS_TSV": os.path.join(CFG["OUTDIR"], "variants_step4.tsv"),
    "COLABFOLD_FASTA": os.path.join(CFG["OUTDIR"], "variants_step5.fasta"),
    "MODELS_DIR": os.path.join(CFG["OUTDIR"], "models_step5"),
})
use_amber = CFG["NUM_RELAX"] > 0
use_templates = False
print("✅ Configuration loaded")


✅ Configuration loaded


In [11]:
#@title 06 - ⬇️ Fetch WT Data (UniProt + AlphaFold)
import os
import requests

print("=" * 60)
print("Fetching Wild-Type Data")
print("=" * 60)

# Ensure output directory exists
os.makedirs(CFG["OUTDIR"], exist_ok=True)

# 1. Fetch FASTA from UniProt
print(f"\n📄 Fetching FASTA for {CFG['UNIPROT_ID']}...")
fasta_url = f"https://rest.uniprot.org/uniprotkb/{CFG['UNIPROT_ID']}.fasta"
wt_fasta = os.path.join(CFG["OUTDIR"], "enzyme_wt.fasta")

try:
    r = requests.get(fasta_url, timeout=60)
    r.raise_for_status()
    with open(wt_fasta, "wb") as f:
        f.write(r.content)

    # Parse sequence info
    with open(wt_fasta) as f:
        lines = f.readlines()
        header = lines[0].strip()
        sequence = "".join(ln.strip() for ln in lines[1:])

    print(f"   ✅ FASTA saved: {wt_fasta}")
    print(f"   → Length: {len(sequence)} amino acids")
    print(f"   → Preview: {sequence[:60]}...")

except Exception as e:
    raise RuntimeError(f"Failed to fetch FASTA: {e}")

# 2. Fetch AlphaFold DB model (v6 - latest version)
print(f"\n🧬 Fetching AlphaFold DB model {CFG['AFDB_VERSION']}...")
af_version = CFG['AFDB_VERSION']
af_pdb_url = f"https://alphafold.ebi.ac.uk/files/AF-{CFG['UNIPROT_ID']}-F1-model_{af_version}.pdb"
af_pdb = os.path.join(CFG["OUTDIR"], f"AF-{CFG['UNIPROT_ID']}-F1-model_{af_version}.pdb")

try:
    r = requests.get(af_pdb_url, timeout=60)
    r.raise_for_status()
    with open(af_pdb, "wb") as f:
        f.write(r.content)

    # Parse model info
    n_atoms = 0
    plddt_values = []
    with open(af_pdb) as f:
        for line in f:
            if line.startswith("ATOM"):
                n_atoms += 1
                try:
                    plddt = float(line[60:66].strip())
                    plddt_values.append(plddt)
                except:
                    pass

    avg_plddt = sum(plddt_values) / len(plddt_values) if plddt_values else 0

    print(f"   ✅ AlphaFold model saved: {af_pdb}")
    print(f"   → Version: {af_version}")
    print(f"   → Atoms: {n_atoms}")
    print(f"   → Average pLDDT: {avg_plddt:.1f}")
    print(f"   → File size: {os.path.getsize(af_pdb):,} bytes")

except Exception as e:
    raise RuntimeError(f"Failed to fetch AlphaFold model: {e}")

# Store in config
CFG["REFERENCE_STRUCTURE"] = af_pdb
CFG["WT_FASTA"] = wt_fasta
CFG["WT_SEQUENCE"] = sequence

print("\n" + "=" * 60)
print("✅ Wild-type data ready!")
print(f"   → FASTA: {wt_fasta}")
print(f"   → Structure: {af_pdb}")
print("=" * 60)

Fetching Wild-Type Data

📄 Fetching FASTA for P0ABQ4...
   ✅ FASTA saved: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/enzyme_wt.fasta
   → Length: 159 amino acids
   → Preview: MISLIAALAVDRVIGMENAMPWNLPADLAWFKRNTLNKPVIMGRHTWESIGRPLPGRKNI...

🧬 Fetching AlphaFold DB model v6...
   ✅ AlphaFold model saved: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/AF-P0ABQ4-F1-model_v6.pdb
   → Version: v6
   → Atoms: 1268
   → Average pLDDT: 96.6
   → File size: 107,729 bytes

✅ Wild-type data ready!
   → FASTA: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/enzyme_wt.fasta
   → Structure: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/AF-P0ABQ4-F1-model_v6.pdb


In [12]:
#@title 07 - 📊 Preview WT Structure
import os

try:
    import py3Dmol
except Exception:
    py3Dmol = None
    print("⚠️ py3Dmol is not installed; skipping 3D view.")

af_pdb = CFG["REFERENCE_STRUCTURE"]

if py3Dmol is not None:
    v = py3Dmol.view(width=700, height=520)
    with open(af_pdb, 'r') as f:
        v.addModel(f.read(), 'pdb')
    v.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient':'roygb','min':50,'max':90}}})
    v.addStyle({'hetflag': True}, {'stick': {'radius': 0.15}})
    v.zoomTo()
    v.show()

# === SAVE PROTEIN STRUCTURE IMAGE ===
try:
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    import numpy as np
    from pathlib import Path

    print("\n📸 Saving protein structure visualization...")
    pdb_file = Path(CFG["REFERENCE_STRUCTURE"])
    if pdb_file.exists():
        coords, bfactors = [], []
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith('ATOM') and ' CA ' in line:
                    coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    bfactors.append(float(line[60:66]))
        if coords:
            coords = np.array(coords)
            bfactors = np.array(bfactors)
            fig = plt.figure(figsize=(12, 5))
            ax1 = fig.add_subplot(121, projection='3d')
            scatter = ax1.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                                c=bfactors, cmap='RdYlGn', s=20, alpha=0.8)
            ax1.plot(coords[:, 0], coords[:, 1], coords[:, 2], 'gray', alpha=0.3, linewidth=0.5)
            ax1.set_title(f'Wild-Type Structure\n{CFG["UNIPROT_ID"]}', fontweight='bold')
            plt.colorbar(scatter, ax=ax1, shrink=0.5, label='pLDDT')
            ax2 = fig.add_subplot(122)
            ax2.hist(bfactors, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
            ax2.axvline(bfactors.mean(), color='red', linestyle='--', label=f'Mean: {bfactors.mean():.1f}')
            ax2.set_xlabel('pLDDT Score')
            ax2.set_ylabel('Frequency')
            ax2.set_title('Confidence Distribution', fontweight='bold')
            ax2.legend()
            ax2.grid(alpha=0.3)
            plt.tight_layout()
            output_path = Path(CFG["OUTDIR"]) / "protein_structure.png"
            plt.savefig(output_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Saved: {output_path}")
except Exception as e:
    print(f"  ⚠️ Skipped image save: {e}")


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


📸 Saving protein structure visualization...
  ✓ Saved: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/protein_structure.png


In [13]:
#@title 08 - 💊 Fetch Reference Ligand
import os
import subprocess
from pathlib import Path

import requests

OUTDIR = Path(CFG["OUTDIR"])
OUTDIR.mkdir(parents=True, exist_ok=True)

cid = str(CFG["LIGAND_SOURCE"]["id"]).strip()
lig_sdf = OUTDIR / "ligand.sdf"
lig_pdb = OUTDIR / "ligand.pdb"


def _require_obabel():
    from shutil import which
    ob = which("obabel")
    if not ob:
        raise RuntimeError("Open Babel (obabel) not found in PATH")
    return ob


def _fetch_bytes(url: str, timeout: int = 60):
    resp = requests.get(url, timeout=timeout, headers={"User-Agent": "ProteinVariantTriage/1.0"})
    resp.raise_for_status()
    return resp.content, resp.headers.get("content-type", "")


def _looks_like_sdf(payload: bytes) -> bool:
    text = payload.decode("utf-8", errors="ignore")
    if not text.strip():
        return False
    if "V3000" in text[:5000]:
        return "M  V30 BEGIN ATOM" in text and "M  V30 END ATOM" in text
    lines = text.splitlines()
    if len(lines) < 4:
        return False
    if "M  END" not in text and "$$$$" not in text:
        return False
    counts = lines[3]
    try:
        int(counts[:3])
        int(counts[3:6])
        return True
    except Exception:
        return False


def _write_valid_sdf(payload: bytes, path: Path) -> bool:
    path.write_bytes(payload)
    if not path.exists() or path.stat().st_size < 100:
        return False
    return _looks_like_sdf(payload)


def _fetch_pubchem_props(cid_value: str):
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_value}/property/"
        "MolecularWeight,HeavyAtomCount,RotatableBondCount/JSON"
    )
    r = requests.get(url, timeout=60, headers={"User-Agent": "ProteinVariantTriage/1.0"})
    r.raise_for_status()
    props = r.json()["PropertyTable"]["Properties"][0]
    return {
        "mw": float(props.get("MolecularWeight", 0) or 0),
        "heavy": int(props.get("HeavyAtomCount", 0) or 0),
        "rot": int(props.get("RotatableBondCount", 0) or 0),
    }


def _run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        tail = "\n".join((p.stderr or "").splitlines()[-20:])
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}\n{tail}")
    return p


def _looks_protein_like_pdb(path: Path):
    aa = {
        "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
        "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"
    }
    atoms = 0
    aa_atoms = 0
    with open(path) as fh:
        for ln in fh:
            if ln.startswith(("ATOM", "HETATM")):
                atoms += 1
                if len(ln) >= 20 and ln[17:20].strip() in aa:
                    aa_atoms += 1
    frac = (aa_atoms / atoms) if atoms else 0.0
    return atoms, frac


_require_obabel()


# Pre-check CID complexity before downloading/converting.
allow_macro = bool(CFG.get("ALLOW_MACRO_LIGAND", False))
max_heavy = int(CFG.get("LIGAND_MAX_HEAVY_ATOMS", 120))
max_rot = int(CFG.get("LIGAND_MAX_ROTATABLE_BONDS", 32))
max_mw = float(CFG.get("LIGAND_MAX_MW", 1500.0))

try:
    props = _fetch_pubchem_props(cid)
    print(
        f"PubChem properties: MW={props['mw']:.1f}, HeavyAtoms={props['heavy']}, RotBonds={props['rot']}"
    )
    too_large = (
        props["heavy"] > max_heavy
        or props["rot"] > max_rot
        or props["mw"] > max_mw
    )
    if too_large and not allow_macro:
        raise RuntimeError(
            "CID appears macro/peptide-like for Vina small-molecule docking. "
            f"CID={cid}, MW={props['mw']:.1f}, Heavy={props['heavy']}, Rot={props['rot']}. "
            "Pick a small-molecule CID or set CFG['ALLOW_MACRO_LIGAND']=True (not recommended)."
        )
except Exception as e:
    if not allow_macro:
        raise
    print("⚠️ Property pre-check failed (continuing due to ALLOW_MACRO_LIGAND=True):", e)

# 1) Try direct SDF download (3D then 2D)
urls = [
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/SDF?record_type=3d",
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/SDF",
]

download_ok = False
last_err = None
for url in urls:
    try:
        payload, ctype = _fetch_bytes(url)
        if _write_valid_sdf(payload, lig_sdf):
            print(f"✓ Downloaded valid SDF from: {url}")
            download_ok = True
            break
        last_err = f"Invalid SDF payload from {url} (content-type={ctype})"
    except Exception as e:
        last_err = f"{url} -> {e}"

# 2) Fallback: fetch SMILES and build SDF via obabel
if not download_ok:
    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/property/CanonicalSMILES/TXT"
    try:
        smi_bytes, _ = _fetch_bytes(smiles_url)
        smiles = smi_bytes.decode("utf-8", errors="ignore").strip().splitlines()[0].strip()
        if not smiles:
            raise RuntimeError("Empty CanonicalSMILES response")
        _run(["obabel", f"-:{smiles}", "-osdf", "-O", str(lig_sdf), "--gen3d"], cwd=str(OUTDIR))
        if not lig_sdf.exists() or lig_sdf.stat().st_size < 100:
            raise RuntimeError("obabel generated empty SDF from SMILES")
        print("✓ Built ligand.sdf from PubChem CanonicalSMILES fallback")
        download_ok = True
    except Exception as e:
        raise RuntimeError(f"Failed to build ligand reference for CID {cid}. Last download error: {last_err}; SMILES fallback error: {e}")

# 3) Convert SDF -> PDB and validate
_run(["obabel", str(lig_sdf), "-O", str(lig_pdb)])
if not lig_pdb.exists() or lig_pdb.stat().st_size < 100:
    raise RuntimeError(f"ligand.pdb was not created correctly: {lig_pdb}")


atoms, aa_frac = _looks_protein_like_pdb(lig_pdb)
print(f"Ligand PDB atoms={atoms} | amino-acid-like atom fraction={aa_frac:.2f}")
if (atoms > int(CFG.get("LIGAND_MAX_HEAVY_ATOMS", 120)) or aa_frac > 0.35) and not bool(CFG.get("ALLOW_MACRO_LIGAND", False)):
    raise RuntimeError(
        "Ligand PDB looks protein-like or too large for Vina small-molecule docking. "
        f"atoms={atoms}, aa_fraction={aa_frac:.2f}."
    )

print("Saved:", lig_sdf, lig_pdb)



PubChem properties: MW=454.4, HeavyAtoms=33, RotBonds=9
✓ Downloaded valid SDF from: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/126941/SDF?record_type=3d
Ligand PDB atoms=55 | amino-acid-like atom fraction=0.00
Saved: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/ligand.sdf /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/ligand.pdb


In [14]:
#@title 10 - 🧭 Define Docking Box (fpocket + preflight)
import os, json, glob, shutil, subprocess, tempfile, numpy as np, pandas as pd

OUTDIR = CFG["OUTDIR"]
AF_FALLBACK = os.path.join(OUTDIR, f"AF-{CFG['UNIPROT_ID']}-F1-model_v6.pdb")
AF_PDB = CFG.get("REFERENCE_STRUCTURE", os.path.join(OUTDIR, "reference_protein.pdb"))
if (not os.path.exists(AF_PDB)) and os.path.exists(AF_FALLBACK):
    AF_PDB = AF_FALLBACK

POCKET_CSV = os.path.join(OUTDIR, "top_pocket.csv")
BOX_JSON = os.path.join(OUTDIR, "box_params.json")
RECEPTOR_PDBQT = os.path.join(OUTDIR, "receptor.pdbqt")
LIGAND_PDB = os.path.join(OUTDIR, "ligand.pdb")
LIGAND_SDF = os.path.join(OUTDIR, "ligand.sdf")
LIGAND_PDBQT = os.path.join(OUTDIR, "ligand.pdbqt")

MARGIN_ANG = float(CFG.get("BOX_MARGIN_ANG", 6.0))
MIN_BOX_AXIS = 18.0


def _which(cmd):
    from shutil import which
    return which(cmd) is not None


def _run_checked(cmd, cwd=None):
    run = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if run.returncode != 0:
        tail = "\n".join((run.stderr or "").splitlines()[-25:])
        raise RuntimeError(f"Command failed ({run.returncode}): {' '.join(cmd)}\n{tail}")


def _detect_obabel_version():
    try:
        p = subprocess.run(["obabel", "-V"], check=False, capture_output=True, text=True)
        txt = (p.stdout or "") + (p.stderr or "")
        import re
        m = re.search(r"Open Babel\s+(\d+\.\d+\.\d+)", txt)
        return m.group(1) if m else None
    except Exception:
        return None


def _set_babel_env():
    ver = _detect_obabel_version() or "3.1.1"
    candidates = [
        (f"/usr/local/lib/openbabel/{ver}", f"/usr/local/share/openbabel/{ver}"),
        (f"/usr/lib/openbabel/{ver}", f"/usr/share/openbabel/{ver}"),
        (f"/opt/conda/lib/openbabel/{ver}", f"/opt/conda/share/openbabel/{ver}"),
    ]
    for lib, dat in candidates:
        if os.path.isdir(lib):
            os.environ["BABEL_LIBDIR"] = lib
        if os.path.isdir(dat):
            os.environ["BABEL_DATADIR"] = dat
    p = subprocess.run(["obabel", "-L", "formats"], capture_output=True, text=True)
    txt = (p.stdout or "") + (p.stderr or "")
    if "LoadAllPlugins" in txt:
        print("⚠️ Open Babel plugin warning persists:", os.environ.get("BABEL_LIBDIR"), os.environ.get("BABEL_DATADIR"))
    else:
        print("✓ Open Babel plugins available.")


def preflight_prepare_pdbqt(receptor_pdb):
    if not os.path.exists(receptor_pdb):
        raise FileNotFoundError(f"Receptor PDB not found: {receptor_pdb}")
    if not _which("obabel"):
        print("⚠️ OpenBabel (obabel) not found; skipping PDBQT preparation in this cell.")
        return False

    _set_babel_env()

    _run_checked([
        "obabel", receptor_pdb, "-O", RECEPTOR_PDBQT,
        "-xr", "-p", "7.4", "-xh", "--partialcharge", "gasteiger"
    ], cwd=OUTDIR)

    txt = open(RECEPTOR_PDBQT).read(2500)
    if ("ROOT" in txt) or ("BRANCH" in txt):
        raise RuntimeError("Receptor PDBQT contains ROOT/BRANCH (not rigid). Check OpenBabel version/flags.")

    if (not os.path.exists(LIGAND_PDB)) or os.path.getsize(LIGAND_PDB) == 0:
        if os.path.exists(LIGAND_SDF) and os.path.getsize(LIGAND_SDF) > 0:
            _run_checked(["obabel", LIGAND_SDF, "-O", LIGAND_PDB], cwd=OUTDIR)
        else:
            raise FileNotFoundError(f"Ligand source missing: {LIGAND_PDB} and {LIGAND_SDF}")

    _run_checked([
        "obabel", LIGAND_PDB, "-O", LIGAND_PDBQT,
        "-p", "7.4", "--partialcharge", "gasteiger"
    ], cwd=OUTDIR)

    print("✓ receptor_pdbqt:", RECEPTOR_PDBQT, os.path.getsize(RECEPTOR_PDBQT), "bytes")
    print("✓ ligand_pdbqt  :", LIGAND_PDBQT, os.path.getsize(LIGAND_PDBQT), "bytes")
    return True


def _is_hydrogen(ln):
    elem = (ln[76:78].strip() if len(ln) >= 78 else "")
    if elem:
        return elem.upper() == "H"
    return ln[12:16].strip().upper().startswith("H")


def _write_fpocket_input(src_pdb, dst_pdb):
    kept = 0
    has_model = False
    in_first_model = False
    with open(src_pdb) as fi, open(dst_pdb, "w") as fo:
        for raw in fi:
            ln = raw.rstrip("\n")
            rec = ln[:6].strip().upper()
            if rec == "MODEL":
                has_model = True
                if in_first_model:
                    break
                in_first_model = True
                continue
            if rec == "ENDMDL":
                if in_first_model:
                    break
                continue
            if has_model and not in_first_model:
                continue
            if rec != "ATOM":
                continue
            if len(ln) < 54:
                continue
            altloc = ln[16].strip() if len(ln) > 16 else ""
            if altloc and altloc not in {"A", "1"}:
                continue
            if _is_hydrogen(ln):
                continue
            chars = list(ln.ljust(80))
            chars[16] = " "
            if not chars[21].strip():
                chars[21] = "A"
            fo.write("".join(chars).rstrip() + "\n")
            kept += 1
        fo.write("END\n")
    if kept == 0:
        raise RuntimeError(f"No protein atoms kept for fpocket input: {src_pdb}")


def maybe_run_fpocket(pdb_path):
    # Always recompute pocket atoms for the active receptor to avoid stale reuse.
    if os.path.exists(POCKET_CSV):
        try:
            os.remove(POCKET_CSV)
        except Exception:
            pass
    if not _which("fpocket"):
        print("⚠️ fpocket not found; using fallback box selection")
        return

    # Run in a no-space temp directory to avoid fpocket path/basename crashes on Windows-mounted paths.
    tmp = tempfile.mkdtemp(prefix="fpocket_run_", dir="/tmp")
    try:
        fp_in = os.path.join(tmp, "fp_in.pdb")
        fp_out = os.path.join(tmp, "fp_in_out")
        _write_fpocket_input(pdb_path, fp_in)

        print(">>> Running fpocket (pre-docking)...")
        run = subprocess.run(["fpocket", "-f", "fp_in.pdb"], cwd=tmp, capture_output=True, text=True)
        if run.returncode != 0:
            tail = "\n".join((run.stderr or "").splitlines()[-25:])
            raise RuntimeError(f"fpocket returned {run.returncode}. stderr tail:\n{tail}")

        pocket_files = sorted(glob.glob(os.path.join(fp_out, "pockets", "pocket*_atm.pdb")))
        if not pocket_files:
            return

        pocket_file = max(pocket_files, key=lambda p: os.path.getsize(p))
        xs, ys, zs = [], [], []
        with open(pocket_file) as fh:
            for ln in fh:
                if ln.startswith(("ATOM", "HETATM")):
                    xs.append(float(ln[30:38]))
                    ys.append(float(ln[38:46]))
                    zs.append(float(ln[46:54]))
        if xs:
            pd.DataFrame({"x": xs, "y": ys, "z": zs}).to_csv(POCKET_CSV, index=False)
            print("✓ Wrote pocket atoms:", POCKET_CSV)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def centroid_from_pdb_atoms(pdb_path, resid_set=None, chains=None):
    xs, ys, zs = [], [], []
    with open(pdb_path) as fh:
        for ln in fh:
            if not ln.startswith(("ATOM", "HETATM")):
                continue
            if chains:
                ch = (ln[21].strip() or "_")
                if ch not in chains:
                    continue
            resi = int(ln[22:26])
            if (resid_set is not None) and (resi not in resid_set):
                continue
            xs.append(float(ln[30:38]))
            ys.append(float(ln[38:46]))
            zs.append(float(ln[46:54]))
    if not xs:
        raise RuntimeError("No atoms found for centroid selection.")
    return np.array([np.mean(xs), np.mean(ys), np.mean(zs)])


def box_from_points(df_xyz, pad=6.0):
    mins = df_xyz[["x", "y", "z"]].min().values
    maxs = df_xyz[["x", "y", "z"]].max().values
    size = (maxs - mins) + 2 * pad
    center = (maxs + mins) / 2.0
    size = np.maximum(size, np.array([MIN_BOX_AXIS] * 3))
    return center.tolist(), size.tolist()


try:
    preflight_prepare_pdbqt(AF_PDB)
except Exception as e:
    print("Preflight PDBQT step failed (non-fatal):", e)

try:
    if os.path.exists(AF_PDB):
        maybe_run_fpocket(AF_PDB)
except Exception as e:
    print("fpocket pre-step failed (non-fatal):", e)

if os.path.exists(POCKET_CSV) and os.path.getsize(POCKET_CSV) > 0:
    dfp = pd.read_csv(POCKET_CSV)
    center, size, src = *box_from_points(dfp, pad=MARGIN_ANG), "fpocket"
else:
    pocket_txt = os.path.join(OUTDIR, "pocket_residues.txt")
    if os.path.exists(pocket_txt):
        resi = [int(x.strip()) for x in open(pocket_txt) if x.strip().lstrip("-").isdigit()]
        center = centroid_from_pdb_atoms(AF_PDB, resid_set=set(resi)).tolist()
        size = [22.5, 22.5, 22.5]
        src = "pocket_residues.txt"
    else:
        center = centroid_from_pdb_atoms(AF_PDB).tolist()
        size = [30.0, 30.0, 30.0]
        src = "whole_protein_centroid"

with open(BOX_JSON, "w") as fh:
    json.dump({
        "center": center,
        "size": size,
        "source": src,
        "margin": MARGIN_ANG,
        "min_axis": MIN_BOX_AXIS,
    }, fh, indent=2)

print(f"Grid source: {src} | center={tuple(round(x,2) for x in center)} | size={tuple(round(x,1) for x in size)} A")
print("-> Saved:", BOX_JSON)



✓ Open Babel plugins available.
✓ receptor_pdbqt: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/receptor.pdbqt 199603 bytes
✓ ligand_pdbqt  : /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/ligand.pdbqt 4236 bytes
>>> Running fpocket (pre-docking)...
✓ Wrote pocket atoms: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/top_pocket.csv
Grid source: fpocket | center=(3.69, -3.84, 2.11) | size=(37.8, 37.3, 30.7) A
-> Saved: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/box_params.json


In [15]:
#@title 11 - 🚢 Dock Ligand (AutoDock Vina)
import csv
import json
import math
import os
import subprocess
import sys
import tempfile
import textwrap
import time
from pathlib import Path


OUTDIR = Path(CFG.get("OUTDIR", "/content/run_LB"))
OUTDIR.mkdir(parents=True, exist_ok=True)

RECEPTOR_PDBQT = Path(CFG.get("RECEPTOR_PDBQT", OUTDIR / "receptor.pdbqt"))
LIGAND_PDBQT = Path(CFG.get("LIGAND_PDBQT", OUTDIR / "ligand.pdbqt"))
LIGAND_SDF = OUTDIR / "ligand.sdf"
LIGAND_MOL2 = OUTDIR / "ligand.mol2"

# Prefer the box created by Cell 10.
BOX_JSON = Path(CFG.get("BOX_JSON", OUTDIR / "box_params.json"))
if not BOX_JSON.exists():
    alt = OUTDIR / "grid_box.json"
    if alt.exists():
        BOX_JSON = alt
assert BOX_JSON.exists(), f"Missing docking box JSON: {BOX_JSON}"

OUT_PDBQT = OUTDIR / "docked_vina_poses.pdbqt"
OUT_SDF = OUTDIR / "docked_vina_poses.sdf"
OUT_CSV = OUTDIR / "docked_vina_energies.csv"


def _which(cmd):
    from shutil import which
    return which(cmd)


def _run_checked(cmd, env=None):
    p = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if p.returncode != 0:
        tail = "\n".join((p.stderr or "").splitlines()[-25:])
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}\n{tail}")
    return p


def _as_float3(v, name):
    if not isinstance(v, (list, tuple)) or len(v) != 3:
        raise ValueError(f"{name} must be a length-3 list")
    out = []
    for x in v:
        y = float(x)
        if not math.isfinite(y):
            raise ValueError(f"{name} has non-finite value: {x}")
        out.append(y)
    return out


def _clamp_box(size_xyz):
    # Keep map sizes in a practical range to avoid pathological memory/time blowups.
    lo, hi = 16.0, 42.0
    return [max(lo, min(hi, float(v))) for v in size_xyz]


def _ensure_ligand_pdbqt():
    if LIGAND_PDBQT.exists() and LIGAND_PDBQT.stat().st_size > 0:
        return
    obabel = _which("obabel")
    if not obabel:
        raise RuntimeError("Missing ligand.pdbqt and obabel not found to build it")

    if LIGAND_SDF.exists() and LIGAND_SDF.stat().st_size > 0:
        _run_checked([obabel, "-isdf", str(LIGAND_SDF), "-opdbqt", "-O", str(LIGAND_PDBQT), "-h"])
    elif LIGAND_MOL2.exists() and LIGAND_MOL2.stat().st_size > 0:
        _run_checked([obabel, "-imol2", str(LIGAND_MOL2), "-opdbqt", "-O", str(LIGAND_PDBQT), "-h"])
    else:
        raise RuntimeError(f"No ligand source available to build PDBQT: {LIGAND_SDF} / {LIGAND_MOL2}")

    if not LIGAND_PDBQT.exists() or LIGAND_PDBQT.stat().st_size == 0:
        raise RuntimeError(f"Failed to create ligand PDBQT: {LIGAND_PDBQT}")


def _vina_runner_script() -> str:
    return textwrap.dedent(
        """
        import csv
        import json
        from vina import Vina
        import sys

        cfg = json.loads(open(sys.argv[1]).read())

        v = Vina(sf_name=cfg["sf_name"], seed=cfg["seed"], cpu=cfg["cpu"], verbosity=1)
        v.set_receptor(cfg["receptor_pdbqt"])
        v.set_ligand_from_file(cfg["ligand_pdbqt"])
        v.compute_vina_maps(center=cfg["center"], box_size=cfg["size"])
        v.dock(exhaustiveness=cfg["exhaustiveness"], n_poses=cfg["n_poses"])
        v.write_poses(cfg["out_pdbqt"], n_poses=cfg["n_poses"], overwrite=True)

        cols = ["E_total", "E_inter", "E_intra", "E_tors", "E_intra_best"]
        try:
            rows = v.energies(n_poses=cfg["n_poses"], energy_range=cfg["energy_range"])
        except Exception:
            e0 = v.score()
            rows = [[e0[0], e0[1], e0[2], e0[3], float("nan")]]

        with open(cfg["out_csv"], "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(cols)
            for r in rows:
                rr = list(r)[:5]
                while len(rr) < 5:
                    rr.append(float("nan"))
                w.writerow(rr)
        """
    )


def _run_vina_isolated(config, timeout_sec):
    with tempfile.TemporaryDirectory(prefix="vina_run_", dir="/tmp") as td:
        td = Path(td)
        cfg_path = td / "cfg.json"
        run_path = td / "run_vina.py"
        cfg_path.write_text(json.dumps(config, indent=2))
        run_path.write_text(_vina_runner_script())

        proc = subprocess.run(
            [sys.executable, str(run_path), str(cfg_path)],
            capture_output=True,
            text=True,
            timeout=timeout_sec,
        )
        return proc


def _write_energy_csv(v, out_csv, n_poses, energy_range):
    cols = ["E_total", "E_inter", "E_intra", "E_tors", "E_intra_best"]
    try:
        rows = v.energies(n_poses=n_poses, energy_range=energy_range)
    except Exception:
        e0 = v.score()
        rows = [[e0[0], e0[1], e0[2], e0[3], float("nan")]]

    with open(out_csv, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(cols)
        for r in rows:
            rr = list(r)[:5]
            while len(rr) < 5:
                rr.append(float("nan"))
            w.writerow(rr)


def _run_vina_direct(config):
    from vina import Vina

    v = Vina(sf_name=config["sf_name"], seed=config["seed"], cpu=config["cpu"], verbosity=1)
    v.set_receptor(config["receptor_pdbqt"])
    v.set_ligand_from_file(config["ligand_pdbqt"])
    v.compute_vina_maps(center=config["center"], box_size=config["size"])
    v.dock(exhaustiveness=config["exhaustiveness"], n_poses=config["n_poses"])
    v.write_poses(config["out_pdbqt"], n_poses=config["n_poses"], overwrite=True)
    _write_energy_csv(v, config["out_csv"], config["n_poses"], config["energy_range"])


def _inspect_ligand_pdbqt(path: Path):
    atoms = 0
    torsdof = None
    with open(path) as fh:
        for ln in fh:
            if ln.startswith(("ATOM", "HETATM")):
                atoms += 1
            elif ln.startswith("TORSDOF"):
                parts = ln.split()
                if len(parts) >= 2:
                    try:
                        torsdof = int(float(parts[1]))
                    except Exception:
                        torsdof = None
    return atoms, torsdof


# Preconditions
aassert_msg = f"Missing receptor PDBQT: {RECEPTOR_PDBQT}"
assert RECEPTOR_PDBQT.exists() and RECEPTOR_PDBQT.stat().st_size > 0, aassert_msg
_ensure_ligand_pdbqt()

lig_atoms, lig_tors = _inspect_ligand_pdbqt(LIGAND_PDBQT)
max_atoms = int(CFG.get("LIGAND_MAX_HEAVY_ATOMS", 120))
max_tors = int(CFG.get("LIGAND_MAX_ROTATABLE_BONDS", 32))
allow_macro = bool(CFG.get("ALLOW_MACRO_LIGAND", False))
print(f"Ligand complexity guard: atoms={lig_atoms}, torsdof={lig_tors}")
if (lig_atoms > max_atoms or ((lig_tors is not None) and lig_tors > max_tors)) and not allow_macro:
    raise RuntimeError(
        "Ligand too large/flexible for stable Vina run in this pipeline. "
        f"atoms={lig_atoms} (max {max_atoms}), torsdof={lig_tors} (max {max_tors}). "
        "Choose a small-molecule ligand CID/input or set ALLOW_MACRO_LIGAND=True (not recommended)."
    )

# Prevent stale outputs from being mistaken as fresh docking results.
for stale in (OUT_PDBQT, OUT_CSV, OUT_SDF):
    try:
        if stale.exists():
            stale.unlink()
    except Exception:
        pass


def _load_box(path: Path):
    box = json.loads(path.read_text())
    if "center" in box and "size" in box:
        center_xyz = _as_float3(box.get("center"), "box.center")
        size_xyz = _clamp_box(_as_float3(box.get("size"), "box.size"))
        return center_xyz, size_xyz

    # Backward compatibility with legacy grid_box.json schema.
    center_xyz = _as_float3([box.get("center_x"), box.get("center_y"), box.get("center_z")], "box.center_xyz")
    size_xyz = _clamp_box(_as_float3([box.get("size_x"), box.get("size_y"), box.get("size_z")], "box.size_xyz"))
    return center_xyz, size_xyz


center, size = _load_box(BOX_JSON)

cpu_default = max(1, min(2, (os.cpu_count() or 2) // 2))
CPU = int(CFG.get("VINA_CPU", cpu_default))
EXHAUSTIVENESS = int(CFG.get("VINA_EXHAUSTIVENESS", 12))
N_POSES = int(CFG.get("VINA_N_POSES", 8))
ENERGY_RANGE = float(CFG.get("VINA_ENERGY_RANGE", 3.0))
SEED = int(CFG.get("VINA_SEED", 1))
SF_NAME = str(CFG.get("VINA_SF", "vina"))
TIMEOUT_SEC = int(CFG.get("VINA_TIMEOUT_SEC", 1800))

print(
    f"Grid center={tuple(round(x,2) for x in center)} | size={tuple(round(x,1) for x in size)} A "
    f"| cpu={CPU} | exhaustiveness={EXHAUSTIVENESS} | n_poses={N_POSES}"
)

cfg = {
    "sf_name": SF_NAME,
    "seed": SEED,
    "cpu": int(max(1, CPU)),
    "exhaustiveness": int(max(1, EXHAUSTIVENESS)),
    "n_poses": int(max(1, N_POSES)),
    "energy_range": float(ENERGY_RANGE),
    "receptor_pdbqt": str(RECEPTOR_PDBQT),
    "ligand_pdbqt": str(LIGAND_PDBQT),
    "center": [float(x) for x in center],
    "size": [float(x) for x in size],
    "out_pdbqt": str(OUT_PDBQT),
    "out_csv": str(OUT_CSV),
}

RUN_ISOLATED = bool(CFG.get("VINA_RUN_ISOLATED", False))
print("Runner mode:", "isolated-subprocess" if RUN_ISOLATED else "in-process")

start = time.time()
print(f">>> Vina single attempt: cpu={cfg['cpu']} ex={cfg['exhaustiveness']} poses={cfg['n_poses']}")
if RUN_ISOLATED:
    try:
        proc = _run_vina_isolated(cfg, timeout_sec=TIMEOUT_SEC)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"Vina timed out after {TIMEOUT_SEC}s")

    if not (proc.returncode == 0 and OUT_PDBQT.exists() and OUT_PDBQT.stat().st_size > 0):
        tail = "\n".join(((proc.stdout or "") + "\n" + (proc.stderr or "")).splitlines()[-60:])
        if proc.returncode < 0:
            sig = -proc.returncode
            tail = f"Process terminated by signal {sig}\n" + tail
            if sig == 9:
                tail += "\nHint: signal 9 can occur from memory pressure during subprocess launch; set CFG['VINA_RUN_ISOLATED']=False to run in-process."
        raise RuntimeError(f"Vina failed rc={proc.returncode}\n{tail}")
else:
    try:
        _run_vina_direct(cfg)
    except Exception as e:
        raise RuntimeError(f"Vina direct run failed: {e}") from e

if not OUT_PDBQT.exists() or OUT_PDBQT.stat().st_size == 0:
    raise RuntimeError(f"Vina reported success but output is missing: {OUT_PDBQT}")

print("✓ Vina docking succeeded")

obabel = _which("obabel")
if not obabel:
    raise RuntimeError("obabel is required to convert docked PDBQT to SDF")

_run_checked([obabel, "-ipdbqt", str(OUT_PDBQT), "-osdf", "-O", str(OUT_SDF), "-h"])

if not OUT_SDF.exists() or OUT_SDF.stat().st_size == 0:
    raise RuntimeError(f"Failed to produce SDF: {OUT_SDF}")

print(f"✓ Wrote Vina poses: {OUT_PDBQT}")
print(f"✓ Wrote SDF: {OUT_SDF}")
print(f"✓ Energies CSV: {OUT_CSV}")
print(f"⏱️ Elapsed: {time.time()-start:.1f}s")



Ligand complexity guard: atoms=38, torsdof=9
Grid center=(3.69, -3.84, 2.11) | size=(37.8, 37.3, 30.7) A | cpu=2 | exhaustiveness=12 | n_poses=8
Runner mode: in-process
>>> Vina single attempt: cpu=2 ex=12 poses=8
Computing Vina grid ... done.
Performing docking (random seed: 1) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
✓ Vina docking succeeded
✓ Wrote Vina poses: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/docked_vina_poses.pdbqt
✓ Wrote SDF: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/docked_vina_poses.sdf
✓ Energies CSV: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/docked_vina_energies.csv
⏱️ Elapsed: 49.1s


In [16]:
#@title 12 - 🧼 Rescore Poses (GNINA + Clustering)
import os, sys, json, re, math, shutil, stat, subprocess
from pathlib import Path
from urllib.request import Request, urlopen, urlretrieve

# =========
# Config
# =========
CFG = globals().get("CFG", {})
OUTDIR = Path(CFG.get("OUTDIR", "/content/run_LB")); OUTDIR.mkdir(parents=True, exist_ok=True)
RECEPTOR_PDBQT = Path(CFG.get("RECEPTOR_PDBQT", OUTDIR/"receptor.pdbqt"))
VINA_SDF       = Path(CFG.get("VINA_SDF",       OUTDIR/"docked_vina_poses.sdf"))
VINA_CSV       = Path(CFG.get("VINA_CSV",       OUTDIR/"docked_vina_energies.csv"))

RMSD_CUTOFF      = float(CFG.get("RMSD_CUTOFF", 2.0))
TAKE_PER_CLUSTER = int(CFG.get("CLUSTER_TAKE_PER_CLUSTER", 1))

RESCORED_TSV = OUTDIR / "gnina_rescored.tsv"
RESCORED_SDF = OUTDIR / "docked_gnina_rescored.sdf"
TOP_SDF      = OUTDIR / "docked_top.sdf"
REP_CSV      = OUTDIR / "cluster_representatives_gnina.csv"
DOCK_JSON    = OUTDIR / "docking_summary.json"

# =========
# Helpers
# =========
def _run(cmd, check=True, env=None):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    print(">>>", " ".join(map(str, cmd))); print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode})")
    return p


def _have_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
        return r.returncode == 0 and "GPU" in (r.stdout or "")
    except Exception:
        return False


def _list_release_assets(tag="v1.3.2"):
    url = f"https://api.github.com/repos/gnina/gnina/releases/tags/{tag}"
    req = Request(url, headers={"Accept": "application/vnd.github+json", "User-Agent": "curl/8"})
    with urlopen(req, timeout=30) as r:
        data = json.load(r)
    return [(a["name"], a["browser_download_url"], int(a.get("size", 0))) for a in data.get("assets", [])]


def _select_asset(assets, want_gpu, force_cpu=False):
    names = [(n.lower(), n, u, s) for n, u, s in assets]
    if force_cpu:
        for nl, n, u, s in names:
            if "cpu" in nl and n.startswith("gnina"):
                return n, u, s
    if want_gpu and not force_cpu:
        for nl, n, u, s in names:
            if "cuda" in nl and n.startswith("gnina"):
                return n, u, s
    for nl, n, u, s in names:
        if n.startswith("gnina") and "cuda" not in nl:
            return n, u, s
    for n, u, s in assets:
        if n.startswith("gnina"):
            return n, u, s
    return (None, None, 0)


def _guess_cudnn_dirs():
    dirs = []
    # Active env + common conda roots.
    for root in [os.environ.get("CONDA_PREFIX"), sys.prefix, str(Path.home()/"miniconda"), str(Path.home()/"miniconda3")]:
        if not root:
            continue
        p = Path(root)
        if (p/"lib").is_dir():
            dirs.append(str(p/"lib"))
        sp = p/"lib"/f"python{sys.version_info.major}.{sys.version_info.minor}"/"site-packages"/"nvidia"/"cudnn"/"lib"
        if sp.is_dir():
            dirs.append(str(sp))

    # Include nearby env libs (fast shallow glob).
    for base in [Path.home()/"miniconda"/"envs", Path.home()/"miniconda3"/"envs"]:
        if base.is_dir():
            for envlib in base.glob("*/lib"):
                if envlib.is_dir():
                    dirs.append(str(envlib))

    # Keep existing LD entries first, then add discovered unique dirs.
    existing = [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]
    out = []
    for d in existing + dirs:
        if d and d not in out:
            out.append(d)
    return out


def _gnina_env():
    env = os.environ.copy()
    ld = _guess_cudnn_dirs()
    if ld:
        env["LD_LIBRARY_PATH"] = ":".join(ld)
    return env


def _download_gnina(force_cpu=False):
    assets = _list_release_assets("v1.3.2")
    name, url, expected_size = _select_asset(assets, _have_gpu(), force_cpu=force_cpu)
    if not url:
        raise RuntimeError("Could not find a suitable GNINA asset.")
    target_dir = Path.home() / "bin"
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / "gnina"
    print(f">> Downloading {name} -> {target}")
    urlretrieve(url, target)
    got = target.stat().st_size
    if expected_size and got != expected_size:
        raise RuntimeError(f"Downloaded GNINA size mismatch: got {got}, expected {expected_size}")
    os.chmod(target, os.stat(target).st_mode | stat.S_IEXEC)
    return str(target)


def _probe_gnina(bin_path, env):
    try:
        p = subprocess.run([bin_path, "--version"], capture_output=True, text=True, env=env)
    except Exception as e:
        return False, f"launch_failed: {e}", ""
    txt = (p.stdout or "") + (p.stderr or "")
    if p.returncode == 0:
        return True, (p.stdout or "").strip(), txt
    if "libcudnn.so.9" in txt:
        return False, "missing_libcudnn9", txt
    return False, f"returncode_{p.returncode}", txt


def _candidate_gnina_bins():
    cands = []
    for c in [
        CFG.get("GNINA_BIN"),
        str(Path.cwd()/"gnina.1.3.2"),
        str(Path.cwd()/"gnina"),
        str((OUTDIR.parent/"gnina.1.3.2")),
        str((OUTDIR.parent/"gnina")),
        shutil.which("gnina"),
        os.environ.get("GNINA_BIN"),
        str(Path.home()/"bin"/"gnina"),
    ]:
        if c and c not in cands and Path(c).exists():
            cands.append(c)
    return cands


def _build_vina_fallback_scores(vina_csv, n_poses_hint):
    import pandas as pd, numpy as np
    if vina_csv.exists() and vina_csv.stat().st_size > 0:
        df = pd.read_csv(vina_csv).reset_index().rename(columns={"index": "pose_idx"})
        if "E_total" in df.columns:
            df["CNNaffinity"] = -df["E_total"].astype(float)
            df["CNNscore"] = (-df["E_total"].astype(float)).rank(method="average", pct=True)
        else:
            df["CNNaffinity"] = np.nan
            df["CNNscore"] = np.nan
        return df
    return pd.DataFrame({
        "pose_idx": list(range(int(n_poses_hint))),
        "CNNaffinity": [np.nan] * int(n_poses_hint),
        "CNNscore": [np.nan] * int(n_poses_hint),
    })


# =========
# 1) Locate GNINA + robust fallback
# =========
GNINA_ENV = _gnina_env()
print("LD_LIBRARY_PATH for GNINA includes", len(GNINA_ENV.get("LD_LIBRARY_PATH", "").split(":")), "entries")

gnina_bin = None
gnina_ok = False
gnina_status = "not_checked"
gnina_detail = ""

for cand in _candidate_gnina_bins():
    ok, status, detail = _probe_gnina(cand, GNINA_ENV)
    if ok:
        gnina_bin, gnina_ok, gnina_status, gnina_detail = cand, True, status, detail
        break
    # Keep last failure in case all candidates fail.
    gnina_bin, gnina_ok, gnina_status, gnina_detail = cand, False, status, detail

if not gnina_ok:
    print(">> GNINA not usable from current candidates; fetching v1.3.2 release...")
    try:
        gnina_bin = _download_gnina(force_cpu=False)
        gnina_ok, gnina_status, gnina_detail = _probe_gnina(gnina_bin, GNINA_ENV)
    except Exception as e:
        gnina_status = f"download_failed: {e}"

if (not gnina_ok) and (gnina_status == "missing_libcudnn9"):
    print("⚠️ GNINA CUDA build requires libcudnn.so.9; trying CPU build...")
    try:
        gnina_bin = _download_gnina(force_cpu=True)
        gnina_ok, gnina_status, gnina_detail = _probe_gnina(gnina_bin, GNINA_ENV)
    except Exception as e:
        gnina_status = f"cpu_download_failed: {e}"

if gnina_ok:
    print("✓ GNINA:", gnina_status)
    print("✓ Using:", gnina_bin)
else:
    print("⚠️ GNINA unavailable:", gnina_status)
    if gnina_detail:
        print("GNINA detail tail\n" + "\n".join(str(gnina_detail).splitlines()[-20:]))
    print("⚠️ Falling back to Vina-only ranking for this step.")


# =========
# 2) Preconditions
# =========
assert RECEPTOR_PDBQT.exists() and RECEPTOR_PDBQT.stat().st_size > 0, f"Missing receptor PDBQT: {RECEPTOR_PDBQT}"
assert VINA_SDF.exists() and VINA_SDF.stat().st_size > 0, f"Missing Vina SDF poses: {VINA_SDF}"


# =========
# 3) Build score table (GNINA or fallback)
# =========
import pandas as pd, numpy as np

if gnina_ok:
    out = _run([gnina_bin, "-r", str(RECEPTOR_PDBQT), "-l", str(VINA_SDF), "--score_only"], env=GNINA_ENV).stdout

    rows, idx = [], 0
    for line in out.splitlines():
        if "CNN" not in line:
            continue
        m_score = re.search(r"CNNscore[:\s]+([\-]?\d+\.?\d*)", line, re.I)
        m_aff   = re.search(r"CNNaffinity[:\s]+([\-]?\d+\.?\d*)", line, re.I)
        if m_score or m_aff:
            rows.append({"pose_idx": idx,
                         "CNNscore": float(m_score.group(1)) if m_score else np.nan,
                         "CNNaffinity": float(m_aff.group(1)) if m_aff else np.nan,
                         "raw": line.strip()})
            idx += 1

    if not rows:
        floats = []
        for line in out.splitlines():
            if "CNN" in line:
                floats += re.findall(r"[-]?\d+\.\d+", line)
        rows = [{"pose_idx": i, "CNNscore": float(s), "CNNaffinity": float(a), "raw": "fallback_parser"}
                for i, (s, a) in enumerate(zip(floats[::2], floats[1::2]))]

    df_scores = pd.DataFrame(rows).reset_index(drop=True)
    if df_scores.empty:
        from rdkit import Chem
        nposes = sum(1 for m in Chem.SDMolSupplier(str(VINA_SDF), removeHs=False, sanitize=False) if m is not None)
        df_scores = _build_vina_fallback_scores(VINA_CSV, nposes)
        gnina_ok = False
        gnina_status = "gnina_parse_failed_fallback_vina"
else:
    from rdkit import Chem
    nposes = sum(1 for m in Chem.SDMolSupplier(str(VINA_SDF), removeHs=False, sanitize=False) if m is not None)
    df_scores = _build_vina_fallback_scores(VINA_CSV, nposes)

if VINA_CSV.exists() and VINA_CSV.stat().st_size > 0:
    try:
        df_vina = pd.read_csv(VINA_CSV).reset_index().rename(columns={"index": "pose_idx"})
        keep_cols = ["pose_idx"] + [c for c in ["E_total", "E_inter", "E_intra", "E_tors", "E_intra_best"] if c in df_vina.columns]
        df_scores = df_scores.merge(df_vina[keep_cols], on="pose_idx", how="left")
    except Exception as e:
        print("(!) Skipping Vina energies merge:", e)

df_scores.to_csv(RESCORED_TSV, index=False)


# =========
# 4) Attach scores + RMSD clustering (RDKit)
# =========
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit.ML.Cluster import Butina
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.warning")
except ImportError:
    _run([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit.ML.Cluster import Butina
    from rdkit import RDLogger
    RDLogger.DisableLog("rdApp.warning")


def _mol_list_from_sdf(sdf_path):
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=False)
    mols = [m for m in suppl if m is not None]
    if not mols:
        raise ValueError(f"No molecules parsed from {sdf_path}")
    out = []
    for m in mols:
        try:
            m = Chem.AddHs(m, addCoords=True)
            Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_KEKULIZE | Chem.SanitizeFlags.SANITIZE_SETAROMATICITY)
        except Exception:
            pass
        out.append(m)
    return out


def _best_rmsd(a, b):
    try:
        return AllChem.GetBestRMS(a, b)
    except Exception:
        if a.GetNumAtoms() == b.GetNumAtoms():
            ca, cb = a.GetConformer(), b.GetConformer()
            s = 0.0
            for i in range(a.GetNumAtoms()):
                pa, pb = ca.GetAtomPosition(i), cb.GetAtomPosition(i)
                s += (pa.x - pb.x) ** 2 + (pa.y - pb.y) ** 2 + (pa.z - pb.z) ** 2
            return (s / a.GetNumAtoms()) ** 0.5
        return 1e6


def _cluster(mols, cutoff):
    dists = []
    n = len(mols)
    for i in range(1, n):
        for j in range(i):
            dists.append(_best_rmsd(mols[i], mols[j]))
    clus = Butina.ClusterData(dists, nPts=n, distThresh=cutoff, isDistData=True)
    return sorted(clus, key=len, reverse=True)


def _write_sdf(mols, path):
    w = Chem.SDWriter(str(path))
    for m in mols:
        w.write(m)
    w.close()


def _pK_to_Kd_nM(pK):
    try:
        return 10 ** (-float(pK)) * 1e9
    except Exception:
        return float("nan")


mols = _mol_list_from_sdf(VINA_SDF)
for i, m in enumerate(mols):
    m.SetProp("pose_idx", str(i))
    if i < len(df_scores):
        r = df_scores.iloc[i]
        if not pd.isna(r.get("CNNscore", np.nan)):
            m.SetProp("CNNscore", f"{float(r['CNNscore']):.6f}")
        if not pd.isna(r.get("CNNaffinity", np.nan)):
            m.SetProp("CNNaffinity", f"{float(r['CNNaffinity']):.6f}")
            m.SetProp("Kd_nM", f"{_pK_to_Kd_nM(r['CNNaffinity']):.1f}")
        for col in ["E_total", "E_inter", "E_intra", "E_tors", "E_intra_best"]:
            if col in df_scores.columns and not pd.isna(r.get(col, np.nan)):
                m.SetProp(col, f"{float(r[col]):.6f}")

_write_sdf(mols, RESCORED_SDF)

clusters = _cluster(mols, RMSD_CUTOFF)
rows = []
for cid, cl in enumerate(clusters):
    for idx in cl:
        def _getf(mm, key):
            try:
                return float(mm.GetProp(key))
            except Exception:
                return float("nan")
        rows.append({
            "cluster_id": cid,
            "pose_idx": idx,
            "CNNaffinity": _getf(mols[idx], "CNNaffinity"),
            "CNNscore": _getf(mols[idx], "CNNscore")
        })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No clusters/poses available after rescoring")

df["sort_key"] = list(zip(-df["CNNaffinity"].fillna(-9e9), -df["CNNscore"].fillna(-9e9)))
df = df.sort_values(["cluster_id", "sort_key"]).reset_index(drop=True)

rep_indices = []
for cid, sub in df.groupby("cluster_id", sort=True):
    rep_indices += sub.head(TAKE_PER_CLUSTER)["pose_idx"].tolist()

rep_df = df[df["pose_idx"].isin(rep_indices)].sort_values(["cluster_id"]).reset_index(drop=True)
rep_df.to_csv(REP_CSV, index=False)

best_row = rep_df.sort_values(["CNNaffinity", "CNNscore"], ascending=False).iloc[0]
primary_idx = int(best_row["pose_idx"])

_write_sdf([mols[primary_idx]], TOP_SDF)

summary = {
    "primary_pose_idx": primary_idx,
    "scoring_backend": "gnina" if gnina_ok else "vina_fallback",
    "gnina_status": gnina_status,
    "gnina_bin": str(gnina_bin) if gnina_bin else None,
    "primary_scores": {
        "CNNaffinity_pK": float(mols[primary_idx].GetProp("CNNaffinity")) if mols[primary_idx].HasProp("CNNaffinity") else None,
        "CNNscore": float(mols[primary_idx].GetProp("CNNscore")) if mols[primary_idx].HasProp("CNNscore") else None,
        "Kd_nM": float(mols[primary_idx].GetProp("Kd_nM")) if mols[primary_idx].HasProp("Kd_nM") else None,
    },
    "clusters": rep_df.to_dict(orient="records"),
    "files": {
        "rescored_table": str(RESCORED_TSV),
        "rescored_sdf": str(RESCORED_SDF),
        "representatives_csv": str(REP_CSV),
        "fpocket_ligand_sdf": str(TOP_SDF),
    }
}
DOCK_JSON.write_text(json.dumps(summary, indent=2))

print("\n== Summary ==")
print(f"Scoring backend: {'GNINA' if gnina_ok else 'Vina fallback'}")
print(f"Poses rescored:   {len(mols)}")
print(f"Clusters found:   {len(clusters)} @ {RMSD_CUTOFF:.1f} A")
print(f"Reps per cluster: {TAKE_PER_CLUSTER}")
print("Primary (for fpocket): pose_idx =", primary_idx)
print(f"Outputs:\n  - {RESCORED_TSV.name}\n  - {RESCORED_SDF.name}\n  - {REP_CSV.name}\n  - {TOP_SDF.name}\n  - {DOCK_JSON.name}")




LD_LIBRARY_PATH for GNINA includes 5 entries
✓ GNINA: gnina v1.3.2 master:f23dd2b   Built Jul  8 2025.
✓ Using: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/gnina.1.3.2
>>> /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/gnina.1.3.2 -r /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/receptor.pdbqt -l /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/docked_vina_poses.sdf --score_only
              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul  8 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /mnt/c/Use

In [17]:
#@title 13 - 🔍 Identify Pocket Residues
import os, re, glob, shutil, tempfile, subprocess
from pathlib import Path
import numpy as np

CFG = CFG
OUTDIR = Path(CFG["OUTDIR"])
af_pdb = Path(CFG.get("REFERENCE_STRUCTURE", OUTDIR / "reference_protein.pdb"))
af_fallback = OUTDIR / f"AF-{CFG['UNIPROT_ID']}-F1-model_v6.pdb"
if (not af_pdb.exists()) and af_fallback.exists():
    af_pdb = af_fallback

docked_sdf = OUTDIR / "docked_top.sdf"

final_pocket_legacy = OUTDIR / "pocket_residues.txt"
final_pocket_chain = OUTDIR / "pocket_residues.chain.txt"

assert docked_sdf.exists() and docked_sdf.stat().st_size > 0, "Missing docked_top.sdf — re-run docking."
assert af_pdb.exists(), f"Reference structure not found: {af_pdb}"

try:
    from rdkit import Chem
except Exception as e:
    raise RuntimeError("RDKit is required for this cell. Install it in your environment first.") from e


def _is_heavy_atom_pdb_line(ln: str) -> bool:
    if not ln.startswith(("ATOM", "HETATM")):
        return False
    elem = ln[76:78].strip()
    if elem:
        return elem.upper() != "H"
    aname = ln[12:16].strip()
    return not aname.upper().startswith("H")


def ligand_heavy_xyz_from_sdf(sdf_path: Path):
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=False)
    m = next((mm for mm in suppl if mm is not None), None)
    if m is None:
        raise ValueError("RDKit could not read SDF")
    conf = m.GetConformer()
    coords = []
    for i, a in enumerate(m.GetAtoms()):
        if a.GetAtomicNum() > 1:
            p = conf.GetAtomPosition(i)
            coords.append((float(p.x), float(p.y), float(p.z)))
    if not coords:
        raise ValueError("No heavy atoms found in SDF")
    return coords


def protein_atoms_from_pdb(pdb_path: Path, heavy_only=True):
    with open(pdb_path) as fh:
        for ln in fh:
            if not ln.startswith("ATOM"):
                continue
            if heavy_only and not _is_heavy_atom_pdb_line(ln):
                continue
            chain = (ln[21].strip() or " ")
            resi = int(ln[22:26])
            icode = (ln[26].strip() or "")
            x = float(ln[30:38]); y = float(ln[38:46]); z = float(ln[46:54])
            yield (chain, resi, icode, x, y, z)


def dist2(a, b):
    dx, dy, dz = (a[0] - b[0]), (a[1] - b[1]), (a[2] - b[2])
    return dx * dx + dy * dy + dz * dz


def residues_within_cutoff(pdb_path: Path, lig_xyz, cutoff=4.5, heavy_only=True, chain_aware=True):
    cut2 = cutoff * cutoff
    pocket = set()
    for chain, resi, icode, x, y, z in protein_atoms_from_pdb(pdb_path, heavy_only=heavy_only):
        for ax, ay, az in lig_xyz:
            if dist2((x, y, z), (ax, ay, az)) <= cut2:
                pocket.add((chain, resi, icode) if chain_aware else resi)
                break
    return sorted(pocket, key=lambda t: (t if isinstance(t, int) else (t[0], t[1], t[2])))


def centroid(coords):
    if not coords:
        return (0.0, 0.0, 0.0)
    xs, ys, zs = zip(*coords)
    n = float(len(coords))
    return (sum(xs) / n, sum(ys) / n, sum(zs) / n)


def read_pocket_atoms(pdbfile: Path, heavy_only=True):
    out = []
    with open(pdbfile) as f:
        for ln in f:
            if not ln.startswith(("ATOM", "HETATM")):
                continue
            if heavy_only and not _is_heavy_atom_pdb_line(ln):
                continue
            chain = (ln[21].strip() or " ")
            resi = int(ln[22:26])
            icode = (ln[26].strip() or "")
            x = float(ln[30:38]); y = float(ln[38:46]); z = float(ln[46:54])
            out.append(((chain, resi, icode), (x, y, z)))
    return out


def _write_sanitized_fpocket_input(src_pdb: Path, dst_pdb: Path):
    kept = 0
    has_model = False
    in_first_model = False
    with open(src_pdb) as fin, open(dst_pdb, "w") as fout:
        for raw in fin:
            ln = raw.rstrip("\n")
            rec = ln[:6].strip().upper()
            if rec == "MODEL":
                has_model = True
                if in_first_model:
                    break
                in_first_model = True
                continue
            if rec == "ENDMDL":
                if in_first_model:
                    break
                continue
            if has_model and not in_first_model:
                continue
            if rec != "ATOM":
                continue
            if len(ln) < 54:
                continue
            altloc = ln[16].strip() if len(ln) > 16 else ""
            if altloc and altloc not in {"A", "1"}:
                continue
            if not _is_heavy_atom_pdb_line(ln):
                continue
            chars = list(ln.ljust(80))
            chars[16] = " "
            if not chars[21].strip():
                chars[21] = "A"
            fout.write("".join(chars).rstrip() + "\n")
            kept += 1
        fout.write("END\n")
    if kept == 0:
        raise RuntimeError("No atoms kept for fpocket input")


# ---------------------- 1) Contact-core from ligand ----------------------
print(">> Building heavy-atom contact core from docked ligand...")
lig_xyz = ligand_heavy_xyz_from_sdf(docked_sdf)
lig_ctr = centroid(lig_xyz)

contact_core = residues_within_cutoff(af_pdb, lig_xyz, cutoff=4.5, heavy_only=True, chain_aware=True)
shell6 = set(residues_within_cutoff(af_pdb, lig_xyz, cutoff=6.0, heavy_only=True, chain_aware=True))
print(f"   Contact-core (<=4.5 A): {len(contact_core)} residues | Shell (<=6.0 A): {len(shell6)} residues")

# ---------------------- 2) fpocket run & selection ----------------------
fpocket_path = shutil.which("fpocket")
used_fpocket = False
fp_res_chain = set()
pocket_choice_dbg = {}

if fpocket_path:
    tmpdir = Path(tempfile.mkdtemp(prefix="fpocket_run_", dir="/tmp"))
    try:
        san_pdb = tmpdir / "fp_in.pdb"
        _write_sanitized_fpocket_input(af_pdb, san_pdb)

        print(f">> Running fpocket on {san_pdb.name} ...")
        run = subprocess.run(["fpocket", "-f", san_pdb.name], cwd=str(tmpdir), capture_output=True, text=True)
        print(f"   fpocket return code: {run.returncode}")

        fp_out = tmpdir / "fp_in_out"
        info_candidates = []
        for cand in [
            fp_out / "fp_in_info.txt",
            fp_out / "pockets_info.txt",
            fp_out / "pocket_info.txt",
        ]:
            if cand.exists():
                info_candidates.append(cand)
        if not info_candidates:
            info_candidates = [Path(x) for x in glob.glob(str(tmpdir / "**/*_info.txt"), recursive=True)]

        meta = {}
        if info_candidates:
            info_path = sorted(info_candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
            cur = None
            with open(info_path) as f:
                for ln in f:
                    m = re.search(r"Pocket\s+(\d+)", ln, re.I)
                    if m:
                        cur = int(m.group(1)); meta.setdefault(cur, {})
                    m = re.search(r"\bDruggability\s*Score\s*:\s*([-+]?\d*\.?\d+)", ln, re.I)
                    if m and cur is not None:
                        meta[cur]["druggability"] = float(m.group(1))
                    m = re.search(r"\bScore\s*:\s*([-+]?\d*\.?\d+)", ln, re.I)
                    if m and cur is not None:
                        meta[cur]["score"] = float(m.group(1))
                    m = re.search(r"\bVolume\s*:\s*([-+]?\d*\.?\d+)", ln, re.I)
                    if m and cur is not None:
                        meta[cur]["volume"] = float(m.group(1))

        pocket_files = []
        for pattern in ["pocket*_atm.pdb", "pocket*.pdb", "pocket*_vert.pdb"]:
            pocket_files.extend(glob.glob(str((fp_out / "pockets") / pattern)))
            pocket_files.extend(glob.glob(str(fp_out / pattern)))

        pocket_ids = sorted({int(m.group(1)) for pf in pocket_files
                             for m in [re.search(r"pocket(\d+)", os.path.basename(pf))] if m})

        pocket_data = []
        for pid in pocket_ids:
            pocket_pdb = None
            for suffix in ["_atm.pdb", ".pdb", "_vert.pdb"]:
                test1 = fp_out / "pockets" / f"pocket{pid}{suffix}"
                test2 = fp_out / f"pocket{pid}{suffix}"
                if test1.exists():
                    pocket_pdb = test1
                    break
                if test2.exists():
                    pocket_pdb = test2
                    break
            if pocket_pdb is None:
                continue

            atoms = read_pocket_atoms(pocket_pdb, heavy_only=True)
            if not atoms:
                continue

            cnt = 0
            for (_, (x, y, z)) in atoms:
                for ax, ay, az in lig_xyz:
                    if dist2((x, y, z), (ax, ay, az)) <= (4.5 * 4.5):
                        cnt += 1
                        break

            pctr = centroid([xyz for _, xyz in atoms])
            dctr = ((pctr[0] - lig_ctr[0]) ** 2 + (pctr[1] - lig_ctr[1]) ** 2 + (pctr[2] - lig_ctr[2]) ** 2) ** 0.5

            resset = {rid for (rid, _) in atoms}
            pocket_data.append({
                "id": pid,
                "file": pocket_pdb,
                "contacts": cnt,
                "center_dist": dctr,
                "druggability": meta.get(pid, {}).get("druggability", -np.inf),
                "score": meta.get(pid, {}).get("score", -np.inf),
                "volume": meta.get(pid, {}).get("volume", np.nan),
                "residues": resset,
            })

        if pocket_data:
            pocket_data.sort(key=lambda p: (-p["contacts"], -p["druggability"], -p["score"], p["center_dist"]))
            best = pocket_data[0]
            used_fpocket = True
            fp_res_chain = best["residues"] & set(shell6)
            pocket_choice_dbg = {k: best[k] for k in ["id", "contacts", "druggability", "score", "center_dist", "volume"]}
            print(f"   ✓ Selected pocket#{best['id']} | contacts={best['contacts']} | "
                  f"drug={best['druggability']:.2f} | score={best['score']:.2f} | dist={best['center_dist']:.2f} A")
        else:
            print("   ⚠️ No fpocket pocket files parsed; continuing with shell fallback.")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
else:
    print(">> fpocket not found — using shell fallback")

# ---------------------- 3) Final pocket assembly ----------------------
contact_core_set = set(contact_core)
if used_fpocket and fp_res_chain:
    final_chain = sorted(contact_core_set | fp_res_chain)
    method = "contact-core ∪ (fpocket∩shell6)"
else:
    # When fpocket fails, keep first-shell contacts + second-shell neighborhood.
    final_chain = sorted(contact_core_set | shell6)
    method = "contact-core ∪ shell6 (no fpocket)"

legacy_resi = sorted({r[1] for r in final_chain})
with open(final_pocket_legacy, "w") as f:
    f.write("\n".join(map(str, legacy_resi)) + "\n")


def _fmt_chain_res(r):
    ch, resi, ic = r
    return f"{ch}:{resi}{ic}" if ic else f"{ch}:{resi}"


with open(final_pocket_chain, "w") as f:
    f.write("\n".join(_fmt_chain_res(r) for r in final_chain) + "\n")

print("\n" + "=" * 64)
print("Pocket Detection Summary")
print(f"  Method used: {method}")
print(f"  Contact-core (<=4.5 A): {len(contact_core)} residues")
print(f"  Shell (<=6.0 A): {len(shell6)} residues")
print(f"  fpocket used: {used_fpocket}")
if used_fpocket and pocket_choice_dbg:
    print(f"  fpocket choice: {pocket_choice_dbg}")
print(f"  Final pocket (chain-aware): {len(final_chain)} residues")
print(f"  Saved legacy (ints): {final_pocket_legacy}")
print(f"  Saved chain-aware:   {final_pocket_chain}")
print("=" * 64)



>> Building heavy-atom contact core from docked ligand...
   Contact-core (<=4.5 A): 21 residues | Shell (<=6.0 A): 30 residues
>> Running fpocket on fp_in.pdb ...
   fpocket return code: 0
   ✓ Selected pocket#1 | contacts=43 | drug=0.64 | score=6.00 | dist=2.89 A

Pocket Detection Summary
  Method used: contact-core ∪ (fpocket∩shell6)
  Contact-core (<=4.5 A): 21 residues
  Shell (<=6.0 A): 30 residues
  fpocket used: True
  fpocket choice: {'id': 1, 'contacts': 43, 'druggability': 0.643, 'score': 6.0, 'center_dist': 2.885889066144763, 'volume': 1655.346}
  Final pocket (chain-aware): 23 residues
  Saved legacy (ints): /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/pocket_residues.txt
  Saved chain-aware:   /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/pocket_residues.chain.txt


In [2]:
#@title 15a+15f - 🐝 Build + Enrich SWARM Context/SiteCards (ProLIF Required)
import os, sys, subprocess
from pathlib import Path

outdir = CFG['OUTDIR']
site_cards = Path(outdir) / 'swarm' / 'site_cards.jsonl'

# Always regenerate canonical inputs first to avoid stale FASTA/PDB/ligand mismatches.
protein_path = CFG.get('REFERENCE_STRUCTURE') or str(Path(outdir) / 'reference_protein.pdb')
ligand_sdf_cfg = CFG.get('LIGAND_SDF') or str(Path(outdir) / 'ligand.sdf')
ligand_pdb_cfg = CFG.get('LIGAND_PDB') or str(Path(outdir) / 'ligand.pdb')

prep_cmd = [
    sys.executable,
    'scripts/swarm/14a_prepare_inputs.py',
    '--outdir', outdir,
    '--protein-source', 'local_pdb',
    '--protein-path', protein_path,
]

uniprot_id = (CFG.get('UNIPROT_ID') or '').strip()
if uniprot_id:
    prep_cmd.extend([
        '--protein-id', uniprot_id,
        '--fasta-source', 'uniprot_accession',
        '--fasta-accession', uniprot_id,
    ])
else:
    prep_cmd.extend(['--fasta-source', 'none'])

if Path(ligand_sdf_cfg).exists() and Path(ligand_sdf_cfg).stat().st_size > 0:
    prep_cmd.extend(['--ligand-source', 'local_sdf', '--ligand-path', ligand_sdf_cfg])
elif Path(ligand_pdb_cfg).exists() and Path(ligand_pdb_cfg).stat().st_size > 0:
    prep_cmd.extend(['--ligand-source', 'local_pdb', '--ligand-path', ligand_pdb_cfg])
else:
    raise FileNotFoundError(f'No valid ligand input found at {ligand_sdf_cfg} or {ligand_pdb_cfg}')

print('>>>', ' '.join(prep_cmd))
subprocess.check_call(prep_cmd)

# Optional API refresh for stronger functional/conservation signals.
# Set SWARM_REFRESH_API_CONTEXT=1 to regenerate swarm_api context with M-CSA + HMMER priors.
if os.environ.get('SWARM_REFRESH_API_CONTEXT', '0').strip() in ('1', 'true', 'True'):
    api_cmd = [
        sys.executable,
        'scripts/swarm/api/build_context_pack.py',
        '--outdir', outdir,
        '--include-mcsa',
        '--include-hmmer',
        '--include-chembl',
    ]
    print('>>>', ' '.join(api_cmd))
    subprocess.check_call(api_cmd)

cmds = [
    [sys.executable, 'scripts/swarm/15a_build_context_pack.py', '--outdir', outdir],
    [sys.executable, 'scripts/swarm/15b_build_site_cards.py', '--outdir', outdir],
    [sys.executable, 'scripts/swarm/15f_enrich_site_cards_prolif.py', '--outdir', outdir, '--max-poses', '30'],
]
for cmd in cmds:
    print('>>>', ' '.join(cmd))
    subprocess.check_call(cmd)

swarm_dir = Path(outdir) / 'swarm'
print('[swarm] Done. Outputs:')
print('  -', swarm_dir / 'context_pack.json')
print('  -', swarm_dir / 'site_cards.jsonl')
print('  -', swarm_dir / 'prolif_features.json')


>>> /home/linjclinjc/miniconda/envs/swarm/bin/python scripts/swarm/14a_prepare_inputs.py --outdir /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4 --protein-source local_pdb --protein-path /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/AF-P0ABQ4-F1-model_v6.pdb --protein-id P0ABQ4 --fasta-source uniprot_accession --fasta-accession P0ABQ4 --ligand-source local_sdf --ligand-path /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/ligand.sdf
Wrote canonical protein PDB: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/reference_protein.pdb
Wrote canonical FASTA: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/enzyme_wt.fasta
Wrote canonical ligand SDF: /mnt/c/Users/linjc/Doc

/home/linjclinjc/miniconda/envs/swarm/lib/python3.10/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)
100%|██████████| 8/8 [00:00<00:00, 89.49it/s]


Wrote ProLIF-enriched site cards: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/site_cards.jsonl
Wrote ProLIF residue features: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/prolif_features.json
Updated residues with ProLIF features: 21
[swarm] Done. Outputs:
  - /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/context_pack.json
  - /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/site_cards.jsonl
  - /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/prolif_features.json


In [3]:
#@title 18-20 - 🧬 Unified Single Recursive SWARM Loop
import os, sys, subprocess, json
from pathlib import Path
outdir = Path(CFG['OUTDIR'])
# Auto-detect available CPUs unless explicitly overridden.
cpu_total = int(CFG.get('SWARM_CPU_TOTAL') or (os.cpu_count() or 2))
cpu_total = max(2, cpu_total)
profile = str(CFG.get('SWARM_EFFICIENCY_PROFILE', 'balanced')).strip().lower()
if profile not in {'fast', 'balanced', 'quality'}:
    profile = 'balanced'
base_args = {
    'start_round': 0,
    'iterations': 6,
    'panel_total': 200,
    'global_panel_budget': 96,
    'final_max_candidates': 100,
    'adaptive': True,
    'voi_cost_threshold': 0.0025,
    'voi_patience': 2,
    'min_iterations': 2,
    'objective_improvement_eps': 0.005,
    'min_budget_fraction_before_voi_stop': 0.75,
    'force_regenerate_proposals': False,
    # objectives and gates
    'min_function': 0.40,
    'min_binding': 0.35,
    'min_stability': 0.40,
    'min_plausibility': 0.25,
    'tau_func_green': 0.60,
    'tau_func_amber': 0.35,
    'max_per_position': 6,
    # mutation gate policy
    'critical_blosum_min': -1,
    'strict_evo_conservation_threshold': 0.95,
    'strict_evo_blosum_min': -1,
    'functional_exploratory_enable': True,
    'functional_exploratory_blosum_min': -2,
    'functional_exploratory_max_extra': 3,
    'functional_exploratory_ligand_shell': 8.0,
    'chemistry_coverage_enable': True,
    'chemistry_coverage_max_extra': 2,
    'chemistry_coverage_blosum_min': -3,
    'chemistry_coverage_distal_enable': True,
    'chemistry_coverage_distal_ligand_shell': 14.0,
    'functional_site_hard_filter': False,
    'near_functional_hard_filter': False,
    'site_card_wt_mismatch_max_frac': 0.10,
    'site_card_wt_mismatch_min_checked': 20,
    'allow_site_card_wt_mismatch': False,
    'dedupe_scope': 'panel',
    'dedupe_lookback_rounds': 2,
    # selector anti-false-negative controls
    'functional_site_binding_floor': 0.35,
    'enable_functional_binding_challenger': False,
    'binding_challenger_frac': 0.0,
    'binding_challenger_min': 0,
    'binding_challenger_max': 24,
    'binding_challenger_min_binding': 0.02,
    'binding_challenger_uncertainty_min': 0.08,
    'binding_challenger_max_signal': 0.35,
    'binding_challenger_min_func': 0.20,
    'min_binding_challenger_selected': 0,
    'chemistry_challenger_frac': 0.0,
    'chemistry_challenger_min': 0,
    'chemistry_challenger_max': 24,
    'chemistry_challenger_min_binding': 0.01,
    'chemistry_challenger_uncertainty_min': 0.05,
    'chemistry_challenger_max_signal': 0.55,
    'min_chemistry_challenger_selected': 0,
    # fast binding delta stage (speed-oriented defaults)
    'fast_binding_check': True,
    'binding_score_all': True,
    'binding_cnn_model': 'fast',
    'binding_cpu_per_job': 1,
    'binding_autobox_add': 6.0,
    'binding_relax_mutants': False,
    'binding_relax_max_iterations': 120,
    'binding_relax_heavy_restraint_k': 25.0,
    'binding_progress_every': 10,
    'binding_max_variants': 0,
}
profile_overrides = {
    'fast': {
        'proposal_total': 300,
        'proposal_max_per_position': 5,
        'ensemble_models': 3,
        'ensemble_max_iter': 350,
        'min_train_samples': 50,
        'ehvi_mc': 16,
        'binding_workers': max(1, min(6, cpu_total - 1)),
    },
    'balanced': {
        'proposal_total': 420,
        'proposal_max_per_position': 4,
        'ensemble_models': 5,
        'ensemble_max_iter': 700,
        'min_train_samples': 80,
        'ehvi_mc': 32,
        'binding_workers': max(1, min(8, cpu_total - 1)),
    },
    'quality': {
        'proposal_total': 560,
        'proposal_max_per_position': 5,
        'ensemble_models': 7,
        'ensemble_max_iter': 900,
        'min_train_samples': 100,
        'ehvi_mc': 48,
        'binding_workers': max(1, min(10, cpu_total - 1)),
    },
}
recursive_args = dict(base_args)
recursive_args.update(profile_overrides[profile])
# User override hook from config.
recursive_args.update(CFG.get('SWARM_RECURSIVE_ARGS', {}))

# Keep args coherent even after overrides.
panel_total = int(recursive_args.get('panel_total', 200))
recursive_args['proposal_total'] = int(max(panel_total + 40, int(recursive_args.get('proposal_total', panel_total + 60))))

requested_binding_cap = int(recursive_args.get('binding_max_variants', 0))
recursive_args['binding_max_variants'] = int(max(0, requested_binding_cap))

recursive_args['binding_workers'] = int(max(1, min(int(recursive_args.get('binding_workers', 4)), max(1, cpu_total - 1))))

if bool(CFG.get('SWARM_CLEAN_RECURSIVE_OUTPUTS', False)):
    swarm_dir = outdir / 'swarm'
    cleanup_globs = [
        'proposals_r*.jsonl',
        'proposals_vespag_r*.jsonl',
        'proposals_*_readable*.tsv',
        'vespag_round*_mutations.csv',
        'vespag_scores_r*.csv',
        'swarm_panel_r*.tsv',
        'swarm_panel_summary_r*.json',
        'recursive_iteration_metrics_r*.json',
        'binding_fastdl_cache_r*.json',
        'binding_fastdl_summary_r*.json',
        'manifest_r*.json',
        'stat_model_diagnostics_r*.json',
        'recursive_adaptive_summary.json',
        'swarm_final_panel.tsv',
        'swarm_final_panel_summary.json',
        'swarm_final_with_janus.tsv',
        'swarm_final_with_janus_summary.json',
        'janus_input.csv',
        'janus_scores.csv',
        'vespag_policy_state.json',
    ]
    removed = 0
    if swarm_dir.exists():
        for pat in cleanup_globs:
            for p in swarm_dir.glob(pat):
                if p.is_file():
                    p.unlink(missing_ok=True)
                    removed += 1
        for p in swarm_dir.glob('binding_fastdl_mutants_r*'):
            if p.is_dir():
                import shutil
                shutil.rmtree(p, ignore_errors=True)
                removed += 1
    print(f'[recursive-clean] removed_artifacts={removed}')

local_gnina = Path.cwd() / 'gnina.1.3.2'
if local_gnina.exists():
    recursive_args.setdefault('gnina_bin', str(local_gnina))

miniconda_lib = Path.home() / 'miniconda' / 'lib'
if miniconda_lib.exists():
    recursive_args.setdefault('binding_ld_library_path', str(miniconda_lib))

cmd = [
    sys.executable,
    'scripts/swarm/20_run_recursive_adaptive_flow.py',
    '--outdir', str(outdir),
]
for k, v in recursive_args.items():
    flag = '--' + str(k).replace('_', '-')
    if isinstance(v, bool):
        if k == 'adaptive':
            cmd.append('--adaptive' if v else '--no-adaptive')
        elif k == 'force_regenerate_proposals':
            if v:
                cmd.append('--force-regenerate-proposals')
        elif k == 'fast_binding_check':
            cmd.append('--fast-binding-check' if v else '--no-fast-binding-check')
        elif k == 'binding_score_all':
            cmd.append('--binding-score-all' if v else '--no-binding-score-all')
        elif k == 'functional_exploratory_enable':
            cmd.append('--functional-exploratory-enable' if v else '--no-functional-exploratory-enable')
        elif k == 'chemistry_coverage_enable':
            cmd.append('--chemistry-coverage-enable' if v else '--no-chemistry-coverage-enable')
        elif k == 'chemistry_coverage_distal_enable':
            cmd.append('--chemistry-coverage-distal-enable' if v else '--no-chemistry-coverage-distal-enable')
        elif k == 'enable_functional_binding_challenger':
            cmd.append('--enable-functional-binding-challenger' if v else '--no-enable-functional-binding-challenger')
        elif k == 'binding_relax_mutants':
            cmd.append('--binding-relax-mutants' if v else '--no-binding-relax-mutants')
        elif k == 'allow_site_card_wt_mismatch':
            if v:
                cmd.append('--allow-site-card-wt-mismatch')
        elif v:
            cmd.append(flag)
        continue
    if v is None:
        continue
    cmd.extend([flag, str(v)])

weights_candidates = [
    Path.cwd() / 'VespaG' / 'model_weights',
    outdir / 'swarm' / 'vespag_runtime' / 'model_weights',
]
weights_dir = next((p for p in weights_candidates if p.exists()), None)
if weights_dir:
    cmd.extend(['--model-weights-dir', str(weights_dir)])

hf_home = Path.home() / '.cache' / 'huggingface'
hf_home.mkdir(parents=True, exist_ok=True)

use_gpu_embeddings = bool(CFG.get('SWARM_USE_GPU_EMBEDDINGS', True))
cuda_available = False
if use_gpu_embeddings:
    try:
        import torch
        cuda_available = bool(torch.cuda.is_available())
    except Exception:
        cuda_available = False
if use_gpu_embeddings and cuda_available:
    cmd.extend(['--hf-home', str(hf_home), '--gpu-embeddings'])
    embedding_device = 'gpu'
else:
    cmd.extend(['--hf-home', str(hf_home), '--cpu-embeddings'])
    embedding_device = 'cpu'

print(f'[recursive] efficiency_profile = {profile} | cpu_total = {cpu_total}')
print(f'[recursive] embedding_device = {embedding_device}')
print(f"[recursive] fast-binding defaults: score_all={recursive_args.get('binding_score_all')} workers={recursive_args.get('binding_workers')} max_variants={recursive_args.get('binding_max_variants')} relax={recursive_args.get('binding_relax_mutants')}")
print('>>>', ' '.join(str(x) for x in cmd))
subprocess.check_call(cmd)

summary_path = outdir / 'swarm' / 'recursive_adaptive_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing recursive summary: {summary_path}')
summary = json.loads(summary_path.read_text())
final_round = int(summary.get('final_round', 0))
last = (summary.get('history') or [{}])[-1]
print('[recursive] run_mode =', summary.get('run_mode'))
print('[recursive] iterations_completed =', summary.get('iterations_completed'))
print('[recursive] rounds_executed =', summary.get('rounds_executed'))
print('[recursive] final_round =', final_round)
print('[recursive] site_card_qc =', summary.get('site_card_qc'))
print('[recursive] global_selected_cumulative =', summary.get('global_selected_cumulative'))
print('[recursive] objective_best =', summary.get('objective_best'))
print('[recursive] last_metrics:', {
    'objective_score': last.get('objective_score'),
    'expected_hvi_max': last.get('expected_hvi_max'),
    'vespag_gate_pass_rate': last.get('vespag_gate_pass_rate'),
    'diversity_ratio': last.get('diversity_ratio'),
    'panel_bind_mean': last.get('panel_bind_mean'),
    'panel_fill': last.get('panel_fill'),
    'panel_total_requested': last.get('panel_total_requested'),
    'selector_selected_binding_challengers': last.get('selector_selected_binding_challengers'),
    'selector_selected_chemistry_challengers': last.get('selected_chemistry_challengers'),
    'global_budget_remaining_after_round': last.get('global_budget_remaining_after_round'),
})
swarm_dir = outdir / 'swarm'
print('[recursive] outputs:')
print('  -', summary_path)
print('  -', swarm_dir / f'proposals_r{final_round}.jsonl')
print('  -', swarm_dir / f'stat_model_diagnostics_r{final_round}.json')
print('  -', swarm_dir / f'swarm_panel_r{final_round}.tsv')


[recursive] efficiency_profile = balanced | cpu_total = 8
[recursive] embedding_device = gpu
[recursive] fast-binding defaults: score_all=True workers=7 max_variants=0 relax=False
>>> /home/linjclinjc/miniconda/envs/swarm/bin/python scripts/swarm/20_run_recursive_adaptive_flow.py --outdir /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4 --start-round 0 --iterations 6 --panel-total 200 --global-panel-budget 96 --final-max-candidates 100 --adaptive --voi-cost-threshold 0.0025 --voi-patience 2 --min-iterations 2 --objective-improvement-eps 0.005 --min-budget-fraction-before-voi-stop 0.75 --min-function 0.4 --min-binding 0.35 --min-stability 0.4 --min-plausibility 0.25 --tau-func-green 0.6 --tau-func-amber 0.35 --max-per-position 6 --critical-blosum-min -1 --strict-evo-conservation-threshold 0.95 --strict-evo-blosum-min -1 --functional-exploratory-enable --functional-exploratory-blosum-min -2 --functional-exploratory-max-extra

In [4]:
#@title 19e - 🏁 Final Janus Scoring (Merged Single Recursive Rounds)
import os, sys, subprocess, json
from pathlib import Path

outdir = CFG['OUTDIR']
outdir_path = Path(outdir)
summary_path = outdir_path / 'swarm' / 'recursive_adaptive_summary.json'

if os.environ.get('SWARM_FINAL_ROUNDS', '').strip():
    FINAL_ROUNDS = os.environ['SWARM_FINAL_ROUNDS'].strip()
elif summary_path.exists():
    summary = json.loads(summary_path.read_text())
    rounds_executed = summary.get('rounds_executed') or []
    if rounds_executed:
        FINAL_ROUNDS = ','.join(str(int(r)) for r in rounds_executed)
    else:
        final_round = int(summary.get('final_round', 0))
        FINAL_ROUNDS = ','.join(str(r) for r in range(0, max(0, final_round) + 1))
else:
    FINAL_ROUNDS = '0'

JANUS_STABILITY_GATE = 'bayes'
JANUS_STABILITY_FDR = os.environ.get('SWARM_JANUS_STABILITY_FDR', '0.1').strip()
DROP_JANUS_OUTLIERS = os.environ.get('SWARM_DROP_JANUS_OUTLIERS', '1').strip() not in ('0', 'false', 'False')

# Janus backend modes: auto | env_python | conda_env | repo_python | custom_cmd
JANUS_BACKEND = os.environ.get('JANUS_BACKEND', 'auto')
JANUS_REPO = Path(os.environ.get('JANUS_REPO', str(Path.cwd() / 'JanusDDG'))).resolve()
JANUS_ENV_NAME = os.environ.get('JANUS_ENV_NAME', 'janus_env')
JANUS_ENV_PYTHON = os.environ.get('JANUS_ENV_PYTHON', f'/home/linjclinjc/miniconda/envs/{JANUS_ENV_NAME}/bin/python')
JANUS_CUSTOM_CMD = os.environ.get('JANUS_CMD', '').strip()


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


if not JANUS_REPO.exists():
    raise FileNotFoundError(f'JANUS_REPO not found: {JANUS_REPO}')

if JANUS_BACKEND == 'custom_cmd':
    if not JANUS_CUSTOM_CMD:
        raise RuntimeError('JANUS_BACKEND=custom_cmd but JANUS_CMD is empty')
    os.environ['JANUS_CMD'] = JANUS_CUSTOM_CMD
elif JANUS_BACKEND == 'env_python' or (JANUS_BACKEND == 'auto' and Path(JANUS_ENV_PYTHON).exists() and not _in_colab()):
    os.environ['JANUS_CMD'] = f'{JANUS_ENV_PYTHON} src/main.py {{input}}'
elif JANUS_BACKEND == 'conda_env' or (JANUS_BACKEND == 'auto' and os.environ.get('CONDA_EXE') and not _in_colab()):
    conda_exe = os.environ.get('CONDA_EXE')
    os.environ['JANUS_CMD'] = f'{conda_exe} run -n {JANUS_ENV_NAME} python src/main.py {{input}}'
elif JANUS_BACKEND in ('repo_python', 'auto'):
    os.environ['JANUS_CMD'] = f'{sys.executable} src/main.py {{input}}'
else:
    raise RuntimeError(f'Unsupported JANUS_BACKEND: {JANUS_BACKEND}')

os.environ['JANUS_REPO'] = str(JANUS_REPO)
print('[janus-final] rounds =', FINAL_ROUNDS)
print('[janus-final] repo   =', os.environ['JANUS_REPO'])
print('[janus-final] cmd    =', os.environ['JANUS_CMD'])

cmd = [
    sys.executable,
    'scripts/swarm/19e_run_final_janus.py',
    '--outdir', outdir,
    '--rounds', FINAL_ROUNDS,
    '--janus-repo', os.environ['JANUS_REPO'],
    '--janus-cmd', os.environ['JANUS_CMD'],
    '--stability-gate', JANUS_STABILITY_GATE,
    '--stability-fdr', JANUS_STABILITY_FDR,
]
cmd.append('--drop-janus-outliers' if DROP_JANUS_OUTLIERS else '--keep-janus-outliers')

print(' '.join(str(x) for x in cmd))
subprocess.check_call(cmd)


[janus-final] rounds = 0,1,2,3,4,5
[janus-final] repo   = /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/JanusDDG
[janus-final] cmd    = /home/linjclinjc/miniconda/envs/janus_env/bin/python src/main.py {input}
/home/linjclinjc/miniconda/envs/swarm/bin/python scripts/swarm/19e_run_final_janus.py --outdir /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4 --rounds 0,1,2,3,4,5 --janus-repo /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/JanusDDG --janus-cmd /home/linjclinjc/miniconda/envs/janus_env/bin/python src/main.py {input} --stability-gate bayes --stability-fdr 0.1 --drop-janus-outliers
Wrote: /mnt/c/Users/linjc/Documents/Visual Code Projects/cmsc408-fa2025-hw9-yosephlin2/ProteinVariantTriage/data/P0ABQ4/swarm/janus_input.csv (46 rows)
> /home/linjclinjc/miniconda/envs/janus_env/bin/python -c import sys, torch; print

0